# 🚀 Despliegue de FastAPI en Azure — Docker + ACR + Web App (Azure CLI)

**Reto proyecto — Desarrollo de Soluciones IA**

Automatización completa del despliegue de una API FastAPI en Azure **sin usar el portal gráfico**: `Dockerfile` optimizado, y tres scripts bash (`build-image.sh`, `push-to-acr.sh`, `deploy-webapp.sh`) que construyen, versionan, suben a **Azure Container Registry** y despliegan en **Azure Web App** con Azure CLI.

Este notebook **genera los archivos reales del proyecto** (los escribe en `azure-fastapi-deployment/`), valida su sintaxis, prueba la API en local y deja un **zip listo para entregar**.

### Mapeo estructura → celdas

```
azure-fastapi-deployment/
├── app.py                  → Celda «app.py» (proporcionado por el instructor)
├── requirements.txt        → Celda «requirements.txt»
├── Dockerfile              → Celda «Dockerfile»          (criterio 1, 25%)
├── build-image.sh          → Celda «build-image.sh»      (criterio 2, 25%)
├── push-to-acr.sh          → Celda «push-to-acr.sh»      (criterio 3, 25%)
├── deploy-webapp.sh        → Celda «deploy-webapp.sh»    (criterio 4, 25%)
└── .dockerignore           → Celda «.dockerignore» (opcional recomendado)
```

### Flujo de uso (en tu máquina, con Docker y Azure CLI)

```bash
# Preparación
docker --version && az --version
az login
az account show

cd azure-fastapi-deployment
chmod +x build-image.sh push-to-acr.sh deploy-webapp.sh

# 1) Build local + doble tag        2) Push a ACR           3) Desplegar Web App
./build-image.sh                    ./push-to-acr.sh        ./deploy-webapp.sh
```

> **Nombres únicos:** antes de ejecutar, edita `ACR_NAME` y `WEBAPP_NAME` (o expórtalos como variables de entorno): deben ser **globalmente únicos** en Azure. Al acabar, borra el resource group para no gastar crédito: `az group delete --name rg-fastapi-deploy --yes --no-wait`.


## 1. `app.py` y `requirements.txt` (proporcionados)

La API base del instructor: FastAPI con CORS abierto y dos endpoints (`/` y `/health`). La celda crea la carpeta del proyecto y escribe ambos archivos tal cual.


In [1]:
from pathlib import Path

PROYECTO = Path("azure-fastapi-deployment")
PROYECTO.mkdir(exist_ok=True)

APP_PY = '''from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="SEO Content API", version="1.0.0")

# CORS middleware (será configurado también en Azure)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def read_root():
    return {"message": "Hello World from Azure!", "status": "running"}

@app.get("/health")
def health_check():
    return {"status": "healthy", "service": "FastAPI on Azure"}
'''

REQUIREMENTS = '''fastapi
uvicorn
'''

(PROYECTO / "app.py").write_text(APP_PY, encoding="utf-8")
(PROYECTO / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
print("✅ Escritos app.py y requirements.txt en", PROYECTO.resolve())


✅ Escritos app.py y requirements.txt en C:\Users\macdu\Documents\Proyectos-Unir-IA\Proyectos_Unir\Diseno_Estrategias_Produccion_Soluciones_IA\reto 2\azure-fastapi-deployment


## 2. `Dockerfile` — criterio 1 (25%)

Puntos clave del criterio y cómo se cumplen:
- **Imagen base apropiada:** `python:3.12-slim` (ligera, oficial, con `pip` listo; cumple el "Python ≥ 3.12" del reto).
- **Caché de capas:** se copia **primero** `requirements.txt` y se instalan dependencias, y **después** el código. Así, cambiar `app.py` no invalida la capa (cara) de dependencias.
- **`--no-cache-dir`**: pip no guarda caché dentro de la imagen → imagen más pequeña.
- **Puerto 8000 expuesto** y **uvicorn** escuchando en `0.0.0.0:8000` (necesario para que el contenedor sea accesible desde fuera).


In [2]:
DOCKERFILE = '''# Imagen base oficial de Python, variante slim (ligera y sin sorpresas de compilacion).
# Alternativa aun mas ligera: python:3.12-alpine. No se usa por defecto porque con FastAPI/
# uvicorn suele requerir instalar toolchain de compilacion (apk add build-base musl-dev) para
# ruedas no precompiladas, lo que complica el build. 'slim' cumple el criterio (slim o alpine).
FROM python:3.12-slim

# Logs sin buffer: se ven en tiempo real en el "Log stream" de Azure
ENV PYTHONUNBUFFERED=1

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# 1) Copiar SOLO requirements.txt e instalar dependencias.
#    Esta capa se cachea: los cambios de codigo no obligan a reinstalar dependencias.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 2) Copiar SOLO el codigo necesario (no todo el contexto), para una imagen mas ligera.
COPY app.py .
# Si en el futuro la app crece a varios modulos, organiza el codigo en src/ y copia solo eso
# (mantiene la imagen pequena y una estructura mas escalable):
# COPY src/ ./src/

# Puerto en el que escucha la API
EXPOSE 8000

# Arranque: uvicorn escuchando en todas las interfaces del contenedor
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

(PROYECTO / "Dockerfile").write_text(DOCKERFILE, encoding="utf-8")
print("✅ Dockerfile escrito:")
print(DOCKERFILE)


✅ Dockerfile escrito:
# Imagen base oficial de Python, variante slim (ligera y sin sorpresas de compilacion).
# Alternativa aun mas ligera: python:3.12-alpine. No se usa por defecto porque con FastAPI/
# uvicorn suele requerir instalar toolchain de compilacion (apk add build-base musl-dev) para
# ruedas no precompiladas, lo que complica el build. 'slim' cumple el criterio (slim o alpine).
FROM python:3.12-slim

# Logs sin buffer: se ven en tiempo real en el "Log stream" de Azure
ENV PYTHONUNBUFFERED=1

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# 1) Copiar SOLO requirements.txt e instalar dependencias.
#    Esta capa se cachea: los cambios de codigo no obligan a reinstalar dependencias.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 2) Copiar SOLO el codigo necesario (no todo el contexto), para una imagen mas ligera.
COPY app.py .
# Si en el futuro la app crece a varios modulos, organiza el codigo en src/ y copia solo eso
# (mantiene la i

## 3. `.dockerignore` (opcional recomendado)

Evita copiar al contexto de build archivos innecesarios (colchón de seguridad para que credenciales, cachés o el propio notebook no acaben dentro de la imagen).


In [3]:
# Nota: NO excluimos el Dockerfile ni los *.sh, para no romper el build cuando se
# usa el contexto completo del proyecto. Solo se excluye lo que no debe ir en la imagen.
DOCKERIGNORE = '''__pycache__/
*.pyc
.env
.git/
.gitignore
*.md
*.ipynb
.vscode/
.idea/
.dockerignore
'''

(PROYECTO / ".dockerignore").write_text(DOCKERIGNORE, encoding="utf-8")
print("✅ .dockerignore escrito.")


✅ .dockerignore escrito.


## 4. `build-image.sh` — criterio 2 (25%)

Build local con **doble tag** (versión semántica `1.0.0` + `latest`), variables reutilizables sobreescribibles por entorno (`VERSION=1.1.0 ./build-image.sh`), mensajes de progreso y `set -euo pipefail` para abortar ante cualquier error.


In [4]:
BUILD_SH = '''#!/usr/bin/env bash
# Script de build y tag de la imagen Docker.
set -euo pipefail

# ── Variables reutilizables (versionamiento semantico) ────────────────────
# Sobrescribibles por entorno:  VERSION=1.1.0 ./build-image.sh
VERSION="${VERSION:-1.0.0}"
IMAGE_NAME="${IMAGE_NAME:-fastapi-app}"
DOCKERFILE="${DOCKERFILE:-Dockerfile}"        # ruta del Dockerfile a usar

if [ "${1:-}" = "-h" ] || [ "${1:-}" = "--help" ]; then
  cat <<'AYUDA'
Uso: ./build-image.sh
Construye la imagen Docker con doble tag (version + latest).
Variables de entorno:
  VERSION      version semantica (por defecto 1.0.0)
  IMAGE_NAME   nombre de la imagen (por defecto fastapi-app)
  DOCKERFILE   ruta del Dockerfile (por defecto Dockerfile)
Ejemplo:  VERSION=1.1.0 ./build-image.sh
AYUDA
  exit 0
fi

# Validar que VERSION tenga formato semantico X.Y.Z (evita tags accidentales)
if ! echo "${VERSION}" | grep -Eq '^[0-9]+\.[0-9]+\.[0-9]+$'; then
  echo "❌ VERSION='${VERSION}' no es semantica (formato esperado X.Y.Z, p. ej. 1.0.0)."
  exit 1
fi

# Comprobacion previa de dependencias
if ! command -v docker >/dev/null 2>&1; then
  echo "❌ Docker no esta instalado o no esta en el PATH. Instala Docker Desktop y reabre la terminal."
  exit 1
fi
if ! docker info >/dev/null 2>&1; then
  echo "❌ El demonio de Docker no responde. Arranca Docker Desktop y espera a que este 'running'."
  exit 1
fi

echo "════════════════════════════════════════════════════"
echo "🔨 Build de la imagen Docker"
echo "   Imagen:  ${IMAGE_NAME}"
echo "   Version: ${VERSION} (+ latest)"
echo "════════════════════════════════════════════════════"

# Build con DOBLE tag en un solo paso: version especifica y latest
docker build -f "${DOCKERFILE}" -t "${IMAGE_NAME}:${VERSION}" -t "${IMAGE_NAME}:latest" .

echo ""
echo "✅ Build completado. Tags creadas:"
docker images "${IMAGE_NAME}" --format "   {{.Repository}}:{{.Tag}}  ({{.Size}})"

echo ""
echo "▶️  Prueba local:  docker run -p 8000:8000 ${IMAGE_NAME}:${VERSION}"
echo "    y abre http://localhost:8000 y http://localhost:8000/health"
'''

ruta = PROYECTO / "build-image.sh"
ruta.write_text(BUILD_SH, encoding="utf-8")
ruta.chmod(0o755)   # equivale a chmod +x
print("✅ build-image.sh escrito y con permisos de ejecución.")


✅ build-image.sh escrito y con permisos de ejecución.


<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
C:\Users\macdu\AppData\Local\Temp\ipykernel_19748\1802331920.py:1: SyntaxWarning: invalid escape sequence '\.'
  BUILD_SH = '''#!/usr/bin/env bash


## 5. `push-to-acr.sh` — criterio 3 (25%)

Crea el **resource group** y el **ACR si no existen** (comprobación con `az acr show`), hace **login**, **re-etiqueta** la imagen local para el registro (`<acr>.azurecr.io/...`, versión y latest), **sube ambas tags** y **lista** lo subido. `--admin-enabled true` deja listas las credenciales que luego usará la Web App.


In [5]:
PUSH_SH = '''#!/usr/bin/env bash
# Script de push de la imagen a Azure Container Registry (ACR).
set -euo pipefail

# ── Variables reutilizables ────────────────────────────────────────────────
VERSION="${VERSION:-1.0.0}"
IMAGE_NAME="${IMAGE_NAME:-fastapi-app}"
ACR_NAME="${ACR_NAME:-tunombreunico}"        # ⚠️ globalmente unico, solo minusculas/numeros
RESOURCE_GROUP="${RESOURCE_GROUP:-rg-fastapi-deploy}"
LOCATION="${LOCATION:-westeurope}"
ACR_SKU="${ACR_SKU:-Basic}"                   # Basic | Standard | Premium

if [ "${1:-}" = "-h" ] || [ "${1:-}" = "--help" ]; then
  cat <<'AYUDA'
Uso: ./push-to-acr.sh [ACR_NAME] [RESOURCE_GROUP] [LOCATION]
Crea el ACR si no existe, hace login y sube la imagen (version + latest).
Variables de entorno: VERSION IMAGE_NAME ACR_NAME RESOURCE_GROUP LOCATION ACR_SKU
Ejemplo:  ACR_NAME=miacr123 ./push-to-acr.sh
AYUDA
  exit 0
fi

# Argumentos posicionales opcionales (ademas de variables de entorno):
#   ./push-to-acr.sh [ACR_NAME] [RESOURCE_GROUP] [LOCATION]
ACR_NAME="${1:-$ACR_NAME}"
RESOURCE_GROUP="${2:-$RESOURCE_GROUP}"
LOCATION="${3:-$LOCATION}"

# Comprobacion de sesion de Azure
if ! az account show >/dev/null 2>&1; then
  echo "❌ No has iniciado sesion en Azure. Ejecuta:  az login"
  exit 1
fi

if [ "${ACR_NAME}" = "tunombreunico" ]; then
  echo "❌ Edita ACR_NAME (o exportalo): debe ser un nombre globalmente unico."
  exit 1
fi

# Comprobacion previa: las imagenes locales deben existir (build previo)
for tag in "${VERSION}" "latest"; do
  if ! docker image inspect "${IMAGE_NAME}:${tag}" >/dev/null 2>&1; then
    echo "❌ No existe la imagen local ${IMAGE_NAME}:${tag}. Ejecuta primero ./build-image.sh"
    exit 1
  fi
done

echo "════════════════════════════════════════════════════"
echo "📦 Push a Azure Container Registry"
echo "   ACR:      ${ACR_NAME}  (grupo ${RESOURCE_GROUP}, ${LOCATION})"
echo "   Imagen:   ${IMAGE_NAME}:${VERSION} y :latest"
echo "════════════════════════════════════════════════════"

# 1) Resource group (az group create es idempotente)
echo "→ Asegurando resource group '${RESOURCE_GROUP}'..."
az group create --name "${RESOURCE_GROUP}" --location "${LOCATION}" --output none

# 2) Crear el ACR solo si no existe
if az acr show --name "${ACR_NAME}" --resource-group "${RESOURCE_GROUP}" --output none 2>/dev/null; then
  echo "→ El ACR '${ACR_NAME}' ya existe; se reutiliza."
else
  echo "→ Creando ACR '${ACR_NAME}' (SKU ${ACR_SKU}, admin habilitado)..."
  az acr create \\
    --name "${ACR_NAME}" \\
    --resource-group "${RESOURCE_GROUP}" \\
    --sku "${ACR_SKU}" \\
    --admin-enabled true \\
    --output none
fi

# 3) Login en el registro
echo "→ Login en ACR..."
az acr login --name "${ACR_NAME}"

# 4) Re-etiquetar la imagen local para el registro
LOGIN_SERVER=$(az acr show --name "${ACR_NAME}" --query loginServer --output tsv)
echo "→ Re-etiquetando para ${LOGIN_SERVER}..."
docker tag "${IMAGE_NAME}:${VERSION}" "${LOGIN_SERVER}/${IMAGE_NAME}:${VERSION}"
docker tag "${IMAGE_NAME}:latest"     "${LOGIN_SERVER}/${IMAGE_NAME}:latest"

# 5) Subir ambas tags
echo "→ Subiendo ${LOGIN_SERVER}/${IMAGE_NAME}:${VERSION} ..."
docker push "${LOGIN_SERVER}/${IMAGE_NAME}:${VERSION}"
echo "→ Subiendo ${LOGIN_SERVER}/${IMAGE_NAME}:latest ..."
docker push "${LOGIN_SERVER}/${IMAGE_NAME}:latest"

# 6) Verificacion: listar imagenes y tags en el ACR
echo ""
echo "✅ Imagenes en el registro '${ACR_NAME}':"
az acr repository list --name "${ACR_NAME}" --output table
echo ""
echo "✅ Tags de '${IMAGE_NAME}':"
az acr repository show-tags --name "${ACR_NAME}" --repository "${IMAGE_NAME}" --output table
echo ""
echo "🔗 Referencias completas de las imagenes subidas:"
echo "   ${LOGIN_SERVER}/${IMAGE_NAME}:${VERSION}"
echo "   ${LOGIN_SERVER}/${IMAGE_NAME}:latest"
'''

ruta = PROYECTO / "push-to-acr.sh"
ruta.write_text(PUSH_SH, encoding="utf-8")
ruta.chmod(0o755)
print("✅ push-to-acr.sh escrito y ejecutable.")


✅ push-to-acr.sh escrito y ejecutable.


## 6. `deploy-webapp.sh` — criterio 4 (25%)

Despliegue end-to-end: **resource group** → **App Service Plan Linux B1** → **Web App con la imagen del ACR** (con credenciales de admin del registro) → **`WEBSITES_PORT=8000`** → **CORS abierto** → **URL final**. Si la Web App ya existe, en vez de fallar **actualiza la imagen** (útil para redesplegar nuevas versiones).


In [6]:
DEPLOY_SH = '''#!/usr/bin/env bash
# Script de despliegue de la Web App a partir de la imagen del ACR.
set -euo pipefail

# ── Variables reutilizables ────────────────────────────────────────────────
VERSION="${VERSION:-1.0.0}"
IMAGE_NAME="${IMAGE_NAME:-fastapi-app}"
ACR_NAME="${ACR_NAME:-tunombreunico}"
RESOURCE_GROUP="${RESOURCE_GROUP:-rg-fastapi-deploy}"
LOCATION="${LOCATION:-westeurope}"
PLAN_NAME="${PLAN_NAME:-plan-fastapi-deploy}"
PLAN_SKU="${PLAN_SKU:-B1}"                    # B1 (Basic) o F1 (Free)
WEBAPP_NAME="${WEBAPP_NAME:-tuwebappunica}"  # ⚠️ globalmente unico (forma parte de la URL)
HEALTH_PATH="${HEALTH_PATH:-/health}"        # ruta del endpoint de salud (parametrizable)

# Opciones de linea de comandos (ademas de variables de entorno):
#   ./deploy-webapp.sh --sku F1 --location westeurope
while [ $# -gt 0 ]; do
  case "$1" in
    --sku)       PLAN_SKU="$2"; shift 2 ;;
    --location)  LOCATION="$2"; shift 2 ;;
    -h|--help)
      echo "Uso: ./deploy-webapp.sh [--sku B1|F1] [--location <region>]"
      echo "Variables: VERSION IMAGE_NAME ACR_NAME RESOURCE_GROUP PLAN_NAME WEBAPP_NAME"
      exit 0 ;;
    *) echo "Opcion desconocida: $1"; exit 1 ;;
  esac
done

# Comprobacion de sesion de Azure
if ! az account show >/dev/null 2>&1; then
  echo "❌ No has iniciado sesion en Azure. Ejecuta:  az login"
  exit 1
fi

if [ "${ACR_NAME}" = "tunombreunico" ] || [ "${WEBAPP_NAME}" = "tuwebappunica" ]; then
  echo "❌ Edita ACR_NAME y WEBAPP_NAME (o exportalos): deben ser globalmente unicos."
  exit 1
fi

# Validar formato del nombre de la Web App (minusculas, numeros y guiones; 2-60 chars)
if ! echo "${WEBAPP_NAME}" | grep -Eq '^[a-z0-9][a-z0-9-]{1,59}$'; then
  echo "❌ WEBAPP_NAME='${WEBAPP_NAME}' no es valido: usa solo minusculas, numeros y guiones"
  echo "   (sin empezar por guion, 2-60 caracteres)."
  exit 1
fi

echo "════════════════════════════════════════════════════"
echo "🚀 Despliegue de Web App desde ACR"
echo "   Web App: ${WEBAPP_NAME}  (plan ${PLAN_NAME}, grupo ${RESOURCE_GROUP})"
echo "   Imagen:  ${ACR_NAME}.azurecr.io/${IMAGE_NAME}:${VERSION}"
echo "════════════════════════════════════════════════════"

# 1) Resource group (idempotente)
echo "→ Asegurando resource group '${RESOURCE_GROUP}'..."
az group create --name "${RESOURCE_GROUP}" --location "${LOCATION}" --output none

# 2) App Service Plan Linux (B1) solo si no existe
if az appservice plan show --name "${PLAN_NAME}" --resource-group "${RESOURCE_GROUP}" --output none 2>/dev/null; then
  echo "→ El plan '${PLAN_NAME}' ya existe; se reutiliza."
else
  echo "→ Creando App Service Plan Linux ${PLAN_SKU} '${PLAN_NAME}'..."
  az appservice plan create \\
    --name "${PLAN_NAME}" \\
    --resource-group "${RESOURCE_GROUP}" \\
    --location "${LOCATION}" \\
    --is-linux \\
    --sku "${PLAN_SKU}" \\
    --output none
fi

# 3) Datos y credenciales del ACR (admin habilitado en push-to-acr.sh)
LOGIN_SERVER=$(az acr show --name "${ACR_NAME}" --query loginServer --output tsv)

# Verificar que la version indicada existe en el ACR (evita desplegar una tag inexistente)
echo "→ Verificando que ${IMAGE_NAME}:${VERSION} existe en el ACR..."
if ! az acr repository show-tags --name "${ACR_NAME}" --repository "${IMAGE_NAME}" --output tsv 2>/dev/null | grep -qx "${VERSION}"; then
  echo "❌ La imagen ${IMAGE_NAME}:${VERSION} no esta en el ACR '${ACR_NAME}'. Ejecuta ./push-to-acr.sh primero."
  exit 1
fi

ACR_USER=$(az acr credential show --name "${ACR_NAME}" --query username --output tsv)
ACR_PASS=$(az acr credential show --name "${ACR_NAME}" --query "passwords[0].value" --output tsv)
IMAGEN_COMPLETA="${LOGIN_SERVER}/${IMAGE_NAME}:${VERSION}"

# 4) Crear la Web App basada en contenedor (o actualizar la imagen si ya existe)
if az webapp show --name "${WEBAPP_NAME}" --resource-group "${RESOURCE_GROUP}" --output none 2>/dev/null; then
  echo "→ La Web App ya existe; actualizando la imagen a ${IMAGEN_COMPLETA}..."
  az webapp config container set \\
    --name "${WEBAPP_NAME}" \\
    --resource-group "${RESOURCE_GROUP}" \\
    --container-image-name "${IMAGEN_COMPLETA}" \\
    --container-registry-url "https://${LOGIN_SERVER}" \\
    --container-registry-user "${ACR_USER}" \\
    --container-registry-password "${ACR_PASS}" \\
    --output none
else
  echo "→ Creando Web App '${WEBAPP_NAME}' con la imagen del ACR..."
  az webapp create \\
    --name "${WEBAPP_NAME}" \\
    --resource-group "${RESOURCE_GROUP}" \\
    --plan "${PLAN_NAME}" \\
    --container-image-name "${IMAGEN_COMPLETA}" \\
    --container-registry-url "https://${LOGIN_SERVER}" \\
    --container-registry-user "${ACR_USER}" \\
    --container-registry-password "${ACR_PASS}" \\
    --output none
fi

# 5) Puerto del contenedor: la Web App enruta el trafico al 8000
echo "→ Configurando WEBSITES_PORT=8000..."
az webapp config appsettings set \\
  --name "${WEBAPP_NAME}" \\
  --resource-group "${RESOURCE_GROUP}" \\
  --settings WEBSITES_PORT=8000 \\
  --output none

# 6) CORS: permitir peticiones desde cualquier origen
echo "→ Habilitando CORS para cualquier origen..."
az webapp cors add \\
  --name "${WEBAPP_NAME}" \\
  --resource-group "${RESOURCE_GROUP}" \\
  --allowed-origins "*" \\
  --output none

# 7) Reiniciar para aplicar y mostrar la URL final
az webapp restart --name "${WEBAPP_NAME}" --resource-group "${RESOURCE_GROUP}" --output none

URL="https://${WEBAPP_NAME}.azurewebsites.net"
echo ""
echo "════════════════════════════════════════════════════"
echo "✅ Despliegue completado."
echo "   🌐 URL:    ${URL}"
echo "   ❤️  Salud: ${URL}${HEALTH_PATH}"
echo "════════════════════════════════════════════════════"

# Comprobacion de estado: sondear el endpoint de salud (el primer arranque tarda en descargar la imagen)
echo "→ Esperando a que la app responda en ${URL}${HEALTH_PATH} (hasta ~3 min)..."
for intento in $(seq 1 18); do
  codigo=$(curl -s -o /dev/null -w "%{http_code}" --max-time 10 "${URL}${HEALTH_PATH}" || echo "000")
  if [ "${codigo}" = "200" ]; then
    echo "✅ La app responde correctamente (HTTP 200 en /health)."
    break
  fi
  echo "   intento ${intento}/18: HTTP ${codigo}; reintentando en 10 s..."
  sleep 10
done
if [ "${codigo}" != "200" ]; then
  echo "⚠️  Aun no responde 200. Es normal en el primer arranque; revisa el Log stream en Azure"
  echo "    o vuelve a probar ${URL}${HEALTH_PATH} en unos minutos."
fi
'''

ruta = PROYECTO / "deploy-webapp.sh"
ruta.write_text(DEPLOY_SH, encoding="utf-8")
ruta.chmod(0o755)
print("✅ deploy-webapp.sh escrito y ejecutable.")


✅ deploy-webapp.sh escrito y ejecutable.


## 7. Validación local

Comprobaciones que sí podemos hacer desde el notebook:
1. **Sintaxis bash** de los 3 scripts (`bash -n`) y permisos de ejecución.
2. **Dockerfile**: presencia de los elementos que pide el criterio (slim, orden de capas, EXPOSE 8000, uvicorn).
3. **La API funciona**: arrancamos `app.py` con uvicorn y probamos `/` y `/health` por HTTP.

El `docker build`/`run` y los comandos `az` se ejecutan en tu máquina (aquí no hay demonio Docker ni sesión de Azure).


In [7]:
import subprocess

# 1) Sintaxis bash y permisos
for script in ("build-image.sh", "push-to-acr.sh", "deploy-webapp.sh"):
    ruta = PROYECTO / script
    resultado = subprocess.run(["bash", "-n", str(ruta)], capture_output=True, text=True)
    estado = "✅ sintaxis OK" if resultado.returncode == 0 else f"❌ {resultado.stderr.strip()}"
    ejecutable = "ejecutable" if ruta.stat().st_mode & 0o111 else "SIN permiso +x"
    print(f"{script:20s} {estado} · {ejecutable}")

# 2) Dockerfile: comprobaciones del criterio
df = (PROYECTO / "Dockerfile").read_text(encoding="utf-8")
comprobaciones = {
    "imagen slim": "python:3.12-slim" in df,
    "requirements antes que el código (caché)": df.index("COPY requirements.txt") < df.index("COPY app.py"),
    "pip sin caché": "--no-cache-dir" in df,
    "EXPOSE 8000": "EXPOSE 8000" in df,
    "uvicorn en 0.0.0.0:8000": '"uvicorn"' in df and '"8000"' in df and "0.0.0.0" in df,
}
for nombre, ok in comprobaciones.items():
    print(f"Dockerfile · {nombre}: {'✅' if ok else '❌'}")
assert all(comprobaciones.values())


build-image.sh       ❌ <3>WSL (881 - Relay) ERROR: CreateProcessCommon:800: execvpe(/bin/bash) failed: No such file or directory · SIN permiso +x
push-to-acr.sh       ❌ <3>WSL (884 - Relay) ERROR: CreateProcessCommon:800: execvpe(/bin/bash) failed: No such file or directory · SIN permiso +x
deploy-webapp.sh     ❌ <3>WSL (887 - Relay) ERROR: CreateProcessCommon:800: execvpe(/bin/bash) failed: No such file or directory · SIN permiso +x
Dockerfile · imagen slim: ✅
Dockerfile · requirements antes que el código (caché): ✅
Dockerfile · pip sin caché: ✅
Dockerfile · EXPOSE 8000: ✅
Dockerfile · uvicorn en 0.0.0.0:8000: ✅


In [8]:
import socket
import threading
import time
import requests
import uvicorn
import importlib.util
import sys

# 3) Arrancar la API del proyecto y probarla por HTTP
PUERTO_LOCAL = 8020

spec = importlib.util.spec_from_file_location("app_proyecto", PROYECTO / "app.py")
modulo = importlib.util.module_from_spec(spec)
sys.modules["app_proyecto"] = modulo
spec.loader.exec_module(modulo)

_srv = None

def _arrancar():
    global _srv
    config = uvicorn.Config(modulo.app, host="127.0.0.1", port=PUERTO_LOCAL, log_level="warning")
    _srv = uvicorn.Server(config)
    _srv.install_signal_handlers = lambda: None
    _srv.run()

def _abierto():
    try:
        with socket.create_connection(("127.0.0.1", PUERTO_LOCAL), timeout=0.5):
            return True
    except OSError:
        return False

if not _abierto():
    threading.Thread(target=_arrancar, daemon=True).start()
    for _ in range(20):
        if _abierto():
            break
        time.sleep(0.25)

r1 = requests.get(f"http://127.0.0.1:{PUERTO_LOCAL}/", timeout=5)
r2 = requests.get(f"http://127.0.0.1:{PUERTO_LOCAL}/health", timeout=5)
print("GET /       ->", r1.status_code, r1.json())
print("GET /health ->", r2.status_code, r2.json())
assert r1.status_code == 200 and r2.json()["status"] == "healthy"
print("\n✅ La API del proyecto funciona. En tu máquina, la prueba equivalente con Docker es:")
print("   docker build -t fastapi-app:1.0.0 .   (o ./build-image.sh)")
print("   docker run -p 8000:8000 fastapi-app:1.0.0")
print("   → http://localhost:8000 y http://localhost:8000/health")


GET /       -> 200 {'message': 'Hello World from Azure!', 'status': 'running'}
GET /health -> 200 {'status': 'healthy', 'service': 'FastAPI on Azure'}

✅ La API del proyecto funciona. En tu máquina, la prueba equivalente con Docker es:
   docker build -t fastapi-app:1.0.0 .   (o ./build-image.sh)
   docker run -p 8000:8000 fastapi-app:1.0.0
   → http://localhost:8000 y http://localhost:8000/health


## 8. Empaquetar para la entrega

Genera `azure-fastapi-deployment.zip` con la estructura completa del proyecto, listo para subir al evaluador (o para hacer `git init` y subirlo a GitHub).


In [9]:
import shutil

# Limpieza previa: no incluir cachés de Python en la entrega
for cache in PROYECTO.rglob("__pycache__"):
    shutil.rmtree(cache, ignore_errors=True)

zip_path = shutil.make_archive("azure-fastapi-deployment", "zip", root_dir=".", base_dir=str(PROYECTO))
print("✅ Proyecto empaquetado:", zip_path)
print("\nContenido:")
for f in sorted(PROYECTO.rglob("*")):
    if f.is_file():
        print(f"   {f}  ({f.stat().st_size} bytes)")


✅ Proyecto empaquetado: c:\Users\macdu\Documents\Proyectos-Unir-IA\Proyectos_Unir\Diseno_Estrategias_Produccion_Soluciones_IA\reto 2\azure-fastapi-deployment.zip

Contenido:
   azure-fastapi-deployment\.dockerignore  (94 bytes)
   azure-fastapi-deployment\app.py  (567 bytes)
   azure-fastapi-deployment\build-image.sh  (2370 bytes)
   azure-fastapi-deployment\deploy-webapp.sh  (7038 bytes)
   azure-fastapi-deployment\Dockerfile  (1256 bytes)
   azure-fastapi-deployment\push-to-acr.sh  (4177 bytes)
   azure-fastapi-deployment\requirements.txt  (18 bytes)


## 9. Chuleta de ejecución real y limpieza

```bash
az login && az account show

cd azure-fastapi-deployment
chmod +x *.sh                       # por si el zip no conserva permisos

# Nombres únicos (o edítalos dentro de los scripts):
export ACR_NAME="acrpedro$(date +%s)"        # solo minúsculas/números
export WEBAPP_NAME="webpedro$(date +%s)"

./build-image.sh                    # build + tags 1.0.0 y latest
./push-to-acr.sh                    # crea ACR si no existe, login, push
./deploy-webapp.sh                  # plan B1 + webapp + puerto + CORS + URL

# Nueva versión más adelante:
VERSION=1.1.0 ./build-image.sh && VERSION=1.1.0 ./push-to-acr.sh && VERSION=1.1.0 ./deploy-webapp.sh
```

**Limpieza al terminar (importante con crédito de estudiante):**
```bash
az group delete --name rg-fastapi-deploy --yes --no-wait
```

Notas:
- `WEBSITES_PORT=8000` es lo que le dice a App Service a qué puerto del contenedor enrutar.
- El CORS queda abierto por partida doble: en el middleware de FastAPI y en la plataforma (`az webapp cors add '*'`), tal como pide el enunciado.
- Si `az acr login` falla en Git Bash de Windows, prueba a arrancar Docker Desktop antes; el login usa el demonio de Docker.
